In [ ]:
# Import basic packages
import pandas as pd
import numpy  as np
import matplotlib.pyplot as plt

## Step 1: Single Data File Loading

In [ ]:
!git clone https://github.com/ljwg3000/UNT_MEEN.git

In [ ]:
import librosa

# load wave file and convert it into array.
data_path = "/content/UNT_MEEN/AI_tutorial/Dataset_sound/Industrial_data_Stethoscope.wav"
Sound, SamplingRate = librosa.load(data_path, sr=None) # You can also convert '.mp3' file

print(SamplingRate)
print(len(Sound))
print(len(Sound)/SamplingRate)

Sound

In [ ]:
from IPython.display import Audio

# Play the sound data
Audio(Sound, rate=SamplingRate)

In [ ]:
time = np.arange(0, len(Sound)/SamplingRate, 1/SamplingRate)

# Plot sound data
plt.figure(figsize=(10,4))
plt.plot(time, Sound)
plt.title("Sound Signal from Warmup Cycle")
plt.xlabel("Time(s)")
plt.ylabel("Amplitude")
plt.grid(alpha=0.5)
plt.show()

## Step 2: Data Segmentation

In [ ]:
# Define variables for segmentation
segment_length = int(1   * SamplingRate)       # segment per 1 second
overlap        = int(0.5 * SamplingRate)       # segment with 0.2 second overlap
step           = int(segment_length - overlap) # step size

print(segment_length, overlap, step)

In [ ]:
segments = []

for i in range(0, len(Sound)- segment_length +1, step):
  segments.append(Sound[i:i+int(segment_length)])

len(segments) # total number of segmented data samples

In [ ]:
for i in range(len(segments)):
  exec(f'Segment_{i+1} = pd.DataFrame(segments[{i}])')

In [ ]:
Segment_283.T

## Step 3: Data Indexing (for Classification)

### Index data by timestamps [second] - Reference
- Class 1 (Tool Change)  : `[12.46, 32.51]`
- Class 2 (Chip Conveyer): `[32.51, 65.57]`    
- Class 3 (X-axis):  `[65.57, 95.30]`
- Class 4 (Y-axis):  `[95.30, 104.36]`
- Class 5 (Z-axis):  `[104.36, 112.12]`
- Class 6 (Spindle):  `[112.12, end]`

In [ ]:
TimeStamp = np.array([12.46, 32.51, 65.57, 95.30, 104.36, 112.12])
DataIndex = TimeStamp * SamplingRate / step
DataIndex

In [ ]:
ClassIdx = [[ 24,  65],
            [ 66, 131],
            [131, 190],
            [191, 208],
            [209, 224],
            [225, len(segments)]]
ClassIdx

In [ ]:
NoOfData_class = [ClassIdx[i][1] - ClassIdx[i][0] for i in range(len(ClassIdx))]
NoOfData_class

In [ ]:
for i in range(len(ClassIdx)):

  Idx = 1
  for j in range(ClassIdx[i][0], ClassIdx[i][1]):
    exec(f'Data_{i+1}_{Idx} = Segment_{j}')
    Idx += 1

In [ ]:
Data_1_41

## Step 4: Feature Extraction

In [ ]:
NoOfClass = len(NoOfData_class)
NoOfClass

In [ ]:
# NoOfSensor  = 1
NoOfFeature = 10

In [ ]:
TimeFeature_C1 = np.zeros((NoOfFeature , NoOfData_class[0]))
TimeFeature_C2 = np.zeros((NoOfFeature , NoOfData_class[1]))
TimeFeature_C3 = np.zeros((NoOfFeature , NoOfData_class[2]))
TimeFeature_C4 = np.zeros((NoOfFeature , NoOfData_class[3]))
TimeFeature_C5 = np.zeros((NoOfFeature , NoOfData_class[4]))
TimeFeature_C6 = np.zeros((NoOfFeature , NoOfData_class[5]))

print(TimeFeature_C1.shape)
print(TimeFeature_C2.shape)
print(TimeFeature_C3.shape)
print(TimeFeature_C4.shape)
print(TimeFeature_C5.shape)
print(TimeFeature_C6.shape)

In [ ]:
# Definition of rms function
def rms(x):
    return np.sqrt(np.mean(x**2))

In [ ]:
import scipy.stats as sp

for i in range(NoOfClass):

  exec(f"temp_TimeFeature = TimeFeature_C{i+1}")

  for j in range(NoOfData_class[i]):

      # Declare temporary data
      exec(f"temp_data = Data_{i+1}_{j+1}")

      temp_TimeFeature[0, j] = np.max(temp_data.iloc[:,0])
      temp_TimeFeature[1, j] = np.min(temp_data.iloc[:,0])
      temp_TimeFeature[2, j] = np.mean(temp_data.iloc[:,0])
      temp_TimeFeature[3, j] = rms(temp_data.iloc[:,0])
      temp_TimeFeature[4, j] = np.var(temp_data.iloc[:,0])
      temp_TimeFeature[5, j] = sp.skew(temp_data.iloc[:,0])
      temp_TimeFeature[6, j] = sp.kurtosis(temp_data.iloc[:,0])
      temp_TimeFeature[7, j] = np.max(temp_data.iloc[:,0])/rms(temp_data.iloc[:,0])
      temp_TimeFeature[8, j] = rms(temp_data.iloc[:,0])/np.mean(np.abs(temp_data.iloc[:,0]))
      temp_TimeFeature[9, j] = np.max(temp_data.iloc[:,0])/np.mean(np.abs(temp_data.iloc[:,0]))

  exec(f"TimeFeature_C{i+1} = temp_TimeFeature")

# Combine Time Features (axis=1)
TimeFeature = np.concatenate([TimeFeature_C1, TimeFeature_C2, TimeFeature_C3,
                              TimeFeature_C4, TimeFeature_C5, TimeFeature_C6] , axis=1)
TimeFeature.shape

In [ ]:
import pywt

MotherWavelet = pywt.Wavelet('haar')   # Mother wavelet
Level   = 8                            # Wavelet decomposition level

In [ ]:
FreqFeature_C1 = np.zeros((NoOfFeature*Level , NoOfData_class[0]))
FreqFeature_C2 = np.zeros((NoOfFeature*Level , NoOfData_class[1]))
FreqFeature_C3 = np.zeros((NoOfFeature*Level , NoOfData_class[2]))
FreqFeature_C4 = np.zeros((NoOfFeature*Level , NoOfData_class[3]))
FreqFeature_C5 = np.zeros((NoOfFeature*Level , NoOfData_class[4]))
FreqFeature_C6 = np.zeros((NoOfFeature*Level , NoOfData_class[5]))

print(FreqFeature_C1.shape)
print(FreqFeature_C2.shape)
print(FreqFeature_C3.shape)
print(FreqFeature_C4.shape)
print(FreqFeature_C5.shape)
print(FreqFeature_C6.shape)

In [ ]:
import scipy.stats as sp

for i in range(NoOfClass):

  exec(f"temp_FreqFeature = FreqFeature_C{i+1}")

  for j in range(NoOfData_class[i]):

      # Declare temporary data
      exec(f"temp_data = Data_{i+1}_{j+1}")

      # Walvelet decomposition
      Coef = pywt.wavedec(temp_data, MotherWavelet, level=Level, axis=0)

      for k in range(Level):
        coef = Coef[Level-k]

        temp_FreqFeature[NoOfFeature*k + 0, j] = np.max(coef[:,0])
        temp_FreqFeature[NoOfFeature*k + 1, j] = np.min(coef[:,0])
        temp_FreqFeature[NoOfFeature*k + 2, j] = np.mean(coef[:,0])
        temp_FreqFeature[NoOfFeature*k + 3, j] = rms(coef[:,0])
        temp_FreqFeature[NoOfFeature*k + 4, j] = np.var(coef[:,0])
        temp_FreqFeature[NoOfFeature*k + 5, j] = sp.skew(coef[:,0])
        temp_FreqFeature[NoOfFeature*k + 6, j] = sp.kurtosis(coef[:,0])
        temp_FreqFeature[NoOfFeature*k + 7, j] = np.max(coef[:,0])/rms(coef[:,0])
        temp_FreqFeature[NoOfFeature*k + 8, j] = rms(coef[:,0])/np.mean(np.abs(coef[:,0]))
        temp_FreqFeature[NoOfFeature*k + 9, j] = np.max(coef[:,0])/np.mean(np.abs(coef[:,0]))

  exec(f"FreqFeature_C{i+1} = temp_FreqFeature")

# Combine Frequency Features (axis=1)
FreqFeature = np.concatenate([FreqFeature_C1, FreqFeature_C2, FreqFeature_C3,
                              FreqFeature_C4, FreqFeature_C5, FreqFeature_C6] , axis=1)
FreqFeature.shape

In [ ]:
Features = np.concatenate([TimeFeature, FreqFeature] , axis=0)
FeatureData = pd.DataFrame(Features)
FeatureData

## Step 5: Feature Selection (ANOVA)

In [ ]:
NoOfData_class_arr = np.array(NoOfData_class)
NoOfData_class_cum = np.cumsum(NoOfData_class_arr)
NoOfData_class_cum

In [ ]:
FeatureData_C1 = FeatureData.iloc[:,                      :NoOfData_class_cum[0]]
FeatureData_C2 = FeatureData.iloc[:, NoOfData_class_cum[0]:NoOfData_class_cum[1]]
FeatureData_C3 = FeatureData.iloc[:, NoOfData_class_cum[1]:NoOfData_class_cum[2]]
FeatureData_C4 = FeatureData.iloc[:, NoOfData_class_cum[2]:NoOfData_class_cum[3]]
FeatureData_C5 = FeatureData.iloc[:, NoOfData_class_cum[3]:NoOfData_class_cum[4]]
FeatureData_C6 = FeatureData.iloc[:, NoOfData_class_cum[4]:NoOfData_class_cum[5]]

print(FeatureData_C1.shape)
print(FeatureData_C2.shape)
print(FeatureData_C3.shape)
print(FeatureData_C4.shape)
print(FeatureData_C5.shape)
print(FeatureData_C6.shape)

In [ ]:
NoOfFeature = FeatureData.shape[0] # Number of feature

P_value = np.zeros((NoOfFeature , 2))

# ANOVA Test (Result: p-value)
for i in np.arange(NoOfFeature):

    ANOVA           = np.array(sp.f_oneway(FeatureData_C1.iloc[i,:] , FeatureData_C2.iloc[i,:],
                                           FeatureData_C3.iloc[i,:] , FeatureData_C4.iloc[i,:],
                                           FeatureData_C5.iloc[i,:] , FeatureData_C6.iloc[i,:]))

    P_value[i,0] = i          # Index of feature
    P_value[i,1] = ANOVA[1]   # P-value

P_value = pd.DataFrame(P_value)  # Convert ANOVA(p-value) result into DataFrame
P_value_Rank = P_value.sort_values([1],ascending=True)  # Sort by P-value in ascending order
P_value_Rank

In [ ]:
import seaborn as sb

# Select the rank of P-value (0 ~ 89)
FeatureRank = 0

# PDF graphs and histograms
class_label=['Tool Change', 'Chip Conveyer', 'Moving X axis', 'Moving Y axis', 'Moving Z axis', 'Spindle Movement']
plt.figure(figsize=(10,5))

for i in range(NoOfClass):
  exec(f"sb.histplot(FeatureData_C{i+1}.iloc[int(P_value_Rank.iloc[FeatureRank,0]),:], label = '{class_label[i]}', kde=True)")

plt.title(f'Feature Rank {FeatureRank+1}', fontsize=20)
plt.legend(loc='best', fontsize=10)
plt.grid(alpha=0.5)
plt.show()

In [ ]:
StartRank = 0
Number    = 20

FeatureSelected = np.zeros((Number,FeatureData.shape[1]))

s = 0
for i in range(StartRank, StartRank+Number):
    index                = int(P_value_Rank.iloc[i-1,0])
    FeatureSelected[s,:] = FeatureData.iloc[index,:].values
    s += 1

FeatureSelected = pd.DataFrame(FeatureSelected)
FeatureSelected